<a href="https://colab.research.google.com/github/HasanAyaz058/flyrank-ml-internship/blob/main/w04_baseline_score(1)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will prioritize content that is **stale and showing a month-over-month position slip**, while requiring enough recent GSC impressions for the item to be actionable.

The two signals checked below are:
- **Staleness** — linked to the refresh/staleness logic from the session.
- **Position slipping** — recent average position is worse than the preceding 30-day window.

The rule is deliberately simple: stale + slipping + visible content receives the highest score. No fitted weights or future/label-derived inputs are used.

Reason codes:
- `STALE_AND_SLIPPING` — stale content with a month-over-month position slip and enough impressions.
- `STALE_ONLY` — stale content without a measured position slip.
- `SLIPPING_ONLY` — position is slipping but the content is not stale.
- `NEITHER` — neither condition is met.

Action labels:
- `REFRESH` — highest-priority content for human review.
- `REVIEW` — one warning signal is present.
- `MONITOR` — no warning signal is present.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import duckdb

# Use a Colab Secret named HF_TOKEN. Do not put the token itself in notebook source.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Add your Hugging Face READ token as a Colab Secret named HF_TOKEN, then run again.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_PREV = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
FACT_CUR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Mid-panel development slice; the final June sample is intentionally excluded.
end_d = "2026-03-31"

# One row per client/content after aggregating two pre-decision 30-day windows.
signals = con.sql(f"""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END AS gsc_avg_position
    FROM {FACT_PREV}
    WHERE gsc_data_available IS TRUE
    UNION ALL
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END AS gsc_avg_position
    FROM {FACT_CUR}
    WHERE gsc_data_available IS TRUE
), agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_impressions ELSE 0 END) AS impressions_current30,
        SUM(CASE WHEN report_date < DATE '2026-03-01' THEN gsc_impressions ELSE 0 END) AS impressions_prev30,
        AVG(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_avg_position END) AS position_current30,
        AVG(CASE WHEN report_date < DATE '2026-03-01' THEN gsc_avg_position END) AS position_prev30
    FROM daily
    GROUP BY 1,2
)
SELECT
    a.*,
    c.content_updated_date,
    c.search_volume,
    c.is_published,
    c.is_deleted
FROM agg a
JOIN {CONTENT} c USING (client_hash_id, content_hash_id)
WHERE c.is_published IS TRUE
  AND c.is_deleted IS FALSE
""").df()

signals["staleness_days"] = (
    pd.Timestamp(end_d) - pd.to_datetime(signals["content_updated_date"])
).dt.days
signals["position_slip"] = signals["position_current30"] - signals["position_prev30"]

# Require usable pre-decision signals.
signals = signals.dropna(subset=["staleness_days", "position_current30", "position_prev30", "impressions_current30"]).copy()
signals = signals[signals["impressions_current30"] >= 100].copy()

# --- Signal check 1: staleness ---
stale_bins = [-1, 90, 180, 365, np.inf]
stale_labels = ["0-90d", "91-180d", "181-365d", "365d+"]
signals["staleness_bucket"] = pd.cut(signals["staleness_days"], bins=stale_bins, labels=stale_labels)
stale_table = (signals.groupby("staleness_bucket", observed=False)
               .agg(n=("content_hash_id","size"), median_impressions=("impressions_current30","median"), median_position_slip=("position_slip","median"))
               .reset_index())
print("=== SIGNAL 1: STALENESS ===")
print(stale_table.to_string(index=False))

# --- Signal check 2: position slipping ---
position_bins = [-np.inf, -2, 0, 2, np.inf]
position_labels = ["improving_2+", "stable_to_improving", "slipping_0-2", "slipping_2+"]
signals["position_bucket"] = pd.cut(signals["position_slip"], bins=position_bins, labels=position_labels)
pos_table = (signals.groupby("position_bucket", observed=False)
             .agg(n=("content_hash_id","size"), median_impressions=("impressions_current30","median"), median_staleness=("staleness_days","median"))
             .reset_index())
print("\n=== SIGNAL 2: POSITION SLIP ===")
print(pos_table.to_string(index=False))

# Evidence-based one-word verdicts printed for the assignment.
# Staleness is CONFIRMED when older buckets show a positive median position slip relative to recent buckets.
recent = stale_table.iloc[0:2]["median_position_slip"].mean()
old = stale_table.iloc[2:]["median_position_slip"].mean()
stale_verdict = "CONFIRMED" if pd.notna(old) and pd.notna(recent) and old > recent else ("OPPOSITE" if pd.notna(old) and pd.notna(recent) and old < recent else "MIXED")
slip_verdict = "CONFIRMED" if (signals["position_slip"] > 2).mean() > 0.05 else "MIXED"
print(f"\nStaleness verdict: {stale_verdict}")
print(f"Position-slip verdict: {slip_verdict}")
print(f"Eligible content items: {len(signals):,}")

## 2. Build the ranked queue (writes the CSV)

The score is a transparent rule: `stale × slipping × recent impressions`. The score is intentionally not fitted. Items with one warning signal remain in the queue with lower priority so the human reviewer can see why they were included.

In [ ]:
# Transparent baseline rule: no fitted weights.
stale = (signals["staleness_days"] >= 180).astype(int)
slipping = (signals["position_slip"] > 2).astype(int)
visible = (signals["impressions_current30"] >= 100).astype(int)

signals["score"] = stale * slipping * visible * signals["impressions_current30"]
signals["reason_code"] = np.select(
    [
        (stale == 1) & (slipping == 1),
        (stale == 1) & (slipping == 0),
        (stale == 0) & (slipping == 1),
    ],
    ["STALE_AND_SLIPPING", "STALE_ONLY", "SLIPPING_ONLY"],
    default="NEITHER",
)
signals["action"] = np.select(
    [signals["reason_code"] == "STALE_AND_SLIPPING", signals["reason_code"].isin(["STALE_ONLY", "SLIPPING_ONLY"])],
    ["REFRESH", "REVIEW"],
    default="MONITOR",
)

queue_cols = [
    "client_hash_id", "content_hash_id", "score", "action", "reason_code",
    "staleness_days", "position_slip", "impressions_current30", "impressions_prev30",
    "position_current30", "position_prev30", "content_updated_date"
]
queue = signals[queue_cols].sort_values(
    ["score", "impressions_current30", "staleness_days"], ascending=[False, False, False]
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
queue = queue[["rank"] + [c for c in queue.columns if c != "rank"]]

out_dir = os.path.join(os.getcwd(), "work", "outputs")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "baseline_action_score.csv")
queue.to_csv(out_path, index=False)

print(f"Ranked queue rows: {len(queue):,}")
print(f"REFRESH: {(queue['action'] == 'REFRESH').sum():,}")
print(f"REVIEW: {(queue['action'] == 'REVIEW').sum():,}")
print(f"MONITOR: {(queue['action'] == 'MONITOR').sum():,}")
print(f"CSV written to: {out_path}")
print("\nTop 10:")
print(queue.head(10).to_string(index=False))

# Small receipt for the repo; the CSV itself is intentionally left out of git by FlyRank design.
receipt = {
    "development_month": "2026-03",
    "rule": "stale AND position_slipping AND visible -> REFRESH priority",
    "score": "stale * slipping * visible * impressions_current30",
    "rows_ranked": int(len(queue)),
    "refresh_rows": int((queue["action"] == "REFRESH").sum()),
    "review_rows": int((queue["action"] == "REVIEW").sum()),
    "monitor_rows": int((queue["action"] == "MONITOR").sum()),
}
with open(os.path.join(out_dir, "baseline_metrics.json"), "w") as f:
    json.dump(receipt, f, indent=2)

## 3. Top-20 review

The review below is generated from the ranked queue. Each row includes the action, reason code, a confidence note, and a concrete condition that would make the pick wrong.

In [ ]:
top20 = queue.head(20).copy()

def confidence(row):
    if row["reason_code"] == "STALE_AND_SLIPPING":
        return "Higher confidence: both rule signals are present."
    if row["reason_code"] == "STALE_ONLY":
        return "Medium confidence: staleness is present but position did not slip."
    if row["reason_code"] == "SLIPPING_ONLY":
        return "Medium confidence: position slipped but content is not stale."
    return "Low confidence: neither warning signal is present."

def wrong_if(row):
    if row["reason_code"] == "STALE_AND_SLIPPING":
        return "Wrong if the position change reflects a temporary SERP mix or measurement issue rather than a content problem."
    if row["reason_code"] == "STALE_ONLY":
        return "Wrong if the page remains healthy despite being old."
    if row["reason_code"] == "SLIPPING_ONLY":
        return "Wrong if the position movement is temporary or caused by a change outside the page."
    return "Wrong if an important problem is not captured by these two signals."

top20["confidence_note"] = top20.apply(confidence, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

review_cols = ["rank", "content_hash_id", "action", "reason_code", "score", "staleness_days", "position_slip", "impressions_current30", "confidence_note", "what_would_make_it_wrong"]
review = top20[review_cols]
print(review.to_string(index=False))

print("\nTop-20 actions:")
print(review["action"].value_counts().to_string())

## 4. Weak picks + leakage check

I will deliberately identify at least one weak pick from the top 20 and verify that the rule uses only pre-decision fields from the March development slice.

In [ ]:
# Weak-pick review: choose the first top-20 item that does not have both warning signals.
weak = review[review["reason_code"] != "STALE_AND_SLIPPING"].head(3)
if len(weak):
    print("Weak picks found in top 20:")
    print(weak[["rank", "content_hash_id", "action", "reason_code", "score", "confidence_note", "what_would_make_it_wrong"]].to_string(index=False))
else:
    print("No weak pick found in the first 20; this should trigger a closer manual review.")

# Leakage guard: list the actual model/rule inputs used.
rule_inputs = [
    "content_updated_date -> staleness_days",
    "gsc_avg_position from March + preceding 30-day window -> position_slip",
    "gsc_impressions from March + preceding 30-day window -> impressions_current30",
    "is_published / is_deleted -> eligibility only",
]
print("\nRule inputs:")
for x in rule_inputs:
    print("-", x)

print("\nLeakage checks:")
print("- Future June sample used: NO")
print("- Label-derived columns used: NO")
print("- Product decision flags used: NO")
print("- Query-table context columns summed: NO")
print("- Development slice: March 2026 mid-panel partition")

assert "is_declining_label" not in signals.columns
assert "trend_pct" not in signals.columns
assert "trend_direction" not in signals.columns
assert queue["rank"].is_monotonic_increasing
assert queue["rank"].iloc[-1] == len(queue)
print("\nLeakage assertions passed.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.